# Corporate Credit Risk (Multiclass Rating)

### Credit Scoring Risk Assessment — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of credit scoring and risk assessment terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the credit scoring and risk assessment problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Credit scoring, probability of default, loss estimation, and stress testing for banking risk management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Banks lend to companies as well as individuals. Corporate credit risk is often expressed as a "Rating" (like S&P or Moody's ratings). Instead of a simple Yes/No, we need to predict which "Bucket" a company falls into.

**Input data used:** Debt-to-Equity ratio, Interest Coverage, Current Ratio, Revenue Growth, and Asset Size.

**What we predict:** Multiclass rating label (`AAA`, `A`, `BB`, `C`).

**ML method used:** Random Forest Classifier (Multiclass).

**Learning goal:** Learn how to handle multiple categories (more than 2) in a classification problem.

**Primary stakeholders:** credit analysts, loan officers, risk managers, and regulatory auditors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Corporate Dataset

We generate 3,000 companies with financial ratios. Better ratios lead to higher ratings.

In [ ]:
n = 3000
rng = RNG

debt_to_equity = rng.uniform(0.1, 5.0, n)
interest_coverage = rng.uniform(0.5, 20.0, n)
revenue_growth = rng.normal(0.05, 0.1, n)
asset_size_log = rng.normal(15, 2, n)

# Logic for Rating Assignment
quality_score = (
    2.0 * interest_coverage - 
    3.0 * debt_to_equity + 
    5.0 * revenue_growth + 
    0.5 * asset_size_log +
    rng.normal(0, 5, n)
)

def assign_rating(score):
    if score > 35: return 'AAA'
    elif score > 20: return 'A'
    elif score > 5: return 'BB'
    else: return 'C'

ratings = [assign_rating(s) for s in quality_score]

df = pd.DataFrame({
    'debt_to_equity': debt_to_equity,
    'interest_coverage': interest_coverage,
    'revenue_growth': revenue_growth,
    'asset_size_log': asset_size_log,
    'rating': ratings
})

print("Rating distribution:")
print(df['rating'].value_counts())
print("\nSample financial data:")
print(df.head())

## Step 4 — Exploratory Data Analysis

EDA

How does Interest Coverage relate to the assigned rating?

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='rating', y='interest_coverage', data=df, order=['AAA', 'A', 'BB', 'C'])
plt.title('Interest Coverage Ratio by Credit Rating')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Distribution plot titled "Interest Coverage Ratio by Credit Rating". The chart highlights distributional differences between groups that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
X = df.drop('rating', axis=1)
y = df['rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=['AAA', 'A', 'BB', 'C'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['AAA', 'A', 'BB', 'C'], yticklabels=['AAA', 'A', 'BB', 'C'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Corporate Ratings')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Heatmap titled "Confusion Matrix: Corporate Ratings" with Predicted on the x-axis and Actual on the y-axis. The heatmap reveals correlation strength and direction between variables.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- The model effectively separates AAA companies (high coverage, low debt) from C-rated companies (low coverage, high debt).
- **Feature Importance:** Interest Coverage and Debt-to-Equity are typically the most critical factors for corporate credit risk.

**Banking Context:** Automated rating models allow banks to monitor their entire corporate loan portfolio in real-time. If a company's financial ratios deteriorate, the model can automatically downgrade the rating, alerting the risk department.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end credit scoring and risk assessment workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Credit models can perpetuate historical discrimination if trained on biased lending data.
- Proxy variables such as zip code or employment type may correlate with protected characteristics.
- Applicants deserve transparent explanations of adverse credit decisions under fair lending regulations.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in credit scoring and risk assessment?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.